In [2]:
!pip install -q openai==1.30.0 httpx==0.27.0 pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.1.52 which is incompatible.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.0.2 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.1.52 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.0 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
mcp 1.27.0 requires httpx>=0.27.1, but you have httpx 0.27.0 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.0 which

In [ ]:
"""
RAGAS EVALUATION — Standard vs Balanced Prompt

METRICS:
  Faithfulness      → answer grounded in contexts? (no hallucination)
  Answer Relevancy  → answer relevant to question?
  Context Relevancy → retrieved contexts relevant to question?

NOTE: We use ContextRelevancy (not ContextPrecision) because
      ContextPrecision requires ground truth labels which we don't have.
"""


# ════════════════════════════════════════════════════════════════
# CELL 1 — Full RAGAS evaluation (paste everything below)
# ════════════════════════════════════════════════════════════════

# ── Step 1: Mount Drive ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print(" Google Drive mounted")

# ── Step 2: Imports ───────────────────────────────────────────
import os
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import httpx

# RAGAS imports — compatible with ragas==0.1.7
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,   # does NOT need ground truth
)

print(" All imports successful")

# ── Step 3: Config ────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
BASELINE_FILE = "/content/drive/MyDrive/arxiv-rag-project/data2/baseline_results.json"
OUTPUT_DIR    = "/content/drive/MyDrive/arxiv-rag-project/data2/cs_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# OpenAI client

client = OpenAI(
    api_key     = os.environ["OPENAI_API_KEY"],
    http_client = httpx.Client(proxies=None)
)
print(f" Config ready | Output → {OUTPUT_DIR}")


# ── Step 4: Load baseline_results.json ───────────────────────
print("\nLoading baseline_results.json ...")
with open(BASELINE_FILE, "r", encoding="utf-8") as f:
    baseline_data = json.load(f)
print(f" Loaded {len(baseline_data)} records")

# verify structure of first record
sample = baseline_data[0]
print(f"  Sample keys: {list(sample.keys())}")
print(f"  Sample query: {sample['query'][:60]}...")
print(f"  Num contexts: {len(sample['contexts'])}")


# ── Step 5: Define prompts ────────────────────────────────────

STANDARD_SYSTEM_PROMPT = """You are a scientific research assistant.
Answer the question based only on the provided context documents.
Be factual and concise."""

BALANCED_SYSTEM_PROMPT = """You are a neutral scientific adjudicator.
Answer the question based on the provided context documents.
If the retrieved context contains conflicting evidence or diverse
geographic perspectives, you MUST represent them proportionally.
Do not favor any single institution or region."""


def build_user_message(query: str, contexts: list) -> str:
    """
    Combine query + contexts into a single user message.
    Truncates each context to 300 chars to stay within token limits.
    """
    ctx_text = "\n\n".join([
        f"[Context {i+1}]: {str(c)[:300]}"
        for i, c in enumerate(contexts)
    ])
    return f"Question: {query}\n\nContexts:\n{ctx_text}"


def get_gpt_answer(query: str, contexts: list, system_prompt: str) -> str:
    """
    Call GPT-4o-mini with given system prompt.
    Returns generated answer string.
    Falls back to empty string on error.
    """
    user_msg = build_user_message(query, contexts)
    try:
        response = client.chat.completions.create(
            model       = "gpt-4o-mini",
            temperature = 0.3,
            messages    = [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_msg},
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"    [WARN] GPT error: {e}")
        return "Answer unavailable due to API error."


# ── Step 6: Generate answers for both prompts ─────────────────
print(f"\nGenerating answers ...")
print(f"  {len(baseline_data)} queries × 2 prompts = {len(baseline_data)*2} GPT calls\n")

records = []

for item in tqdm(baseline_data, desc="Generating GPT answers"):
    query    = item.get("query", "")
    contexts = item.get("contexts", [])

    # ensure contexts is a list of strings
    if not isinstance(contexts, list):
        contexts = [str(contexts)]
    contexts = [str(c) for c in contexts if c]

    if not query or not contexts:
        continue

    # standard prompt answer
    std_answer = get_gpt_answer(query, contexts, STANDARD_SYSTEM_PROMPT)

    # balanced prompt answer
    bal_answer = get_gpt_answer(query, contexts, BALANCED_SYSTEM_PROMPT)

    records.append({
        "query_id"        : item.get("query_id", ""),
        "query"           : query,
        "query_type"      : item.get("query_type", ""),
        "contexts"        : contexts,
        "standard_answer" : std_answer,
        "balanced_answer" : bal_answer,
        "privileged_rate" : item.get("privileged_rate", None),
        "srr"             : item.get("srr", None),
        "spd"             : item.get("spd", None),
        "precision_at_10" : item.get("precision_at_10", None),
        "ndcg_at_10"      : item.get("ndcg_at_10", None),
        "mrr"             : item.get("mrr", None),
    })

print(f"\n Generated {len(records)} answer pairs")

# ── save answers checkpoint (in case RAGAS fails later) ───────
checkpoint_path = os.path.join(OUTPUT_DIR, "ragas_answers_checkpoint.json")
with open(checkpoint_path, "w") as f:
    # save without contexts to keep file small
    checkpoint = [{k: v for k, v in r.items() if k != "contexts"}
                  for r in records]
    json.dump(checkpoint, f, indent=2)
print(f" Answers saved to checkpoint: {checkpoint_path}")


# ── Step 7: Run RAGAS — STANDARD prompt ──────────────────────
print("\nRunning RAGAS — STANDARD prompt ...")

std_dataset = Dataset.from_dict({
    "question" : [r["query"]           for r in records],
    "answer"   : [r["standard_answer"] for r in records],
    "contexts" : [r["contexts"]        for r in records],
})

try:
    std_result = evaluate(
        dataset = std_dataset,
        metrics = [faithfulness, answer_relevancy, context_relevancy],
    )
    std_df = std_result.to_pandas()
    # rename columns to avoid clash with balanced
    std_df = std_df.rename(columns={
        "faithfulness"      : "std_faithfulness",
        "answer_relevancy"  : "std_answer_relevancy",
        "context_relevancy" : "std_context_relevancy",
    })
    print(" Standard RAGAS complete")
    print(f"  Faithfulness      : {std_df['std_faithfulness'].mean():.4f}")
    print(f"  Answer Relevancy  : {std_df['std_answer_relevancy'].mean():.4f}")
    print(f"  Context Relevancy : {std_df['std_context_relevancy'].mean():.4f}")
except Exception as e:
    print(f"[ERROR] Standard RAGAS failed: {e}")
    raise


# ── Step 8: Run RAGAS — BALANCED prompt ──────────────────────
print("\nRunning RAGAS — BALANCED prompt ...")

bal_dataset = Dataset.from_dict({
    "question" : [r["query"]           for r in records],
    "answer"   : [r["balanced_answer"] for r in records],
    "contexts" : [r["contexts"]        for r in records],
})

try:
    bal_result = evaluate(
        dataset = bal_dataset,
        metrics = [faithfulness, answer_relevancy, context_relevancy],
    )
    bal_df = bal_result.to_pandas()
    bal_df = bal_df.rename(columns={
        "faithfulness"      : "bal_faithfulness",
        "answer_relevancy"  : "bal_answer_relevancy",
        "context_relevancy" : "bal_context_relevancy",
    })
    print("✓ Balanced RAGAS complete")
    print(f"  Faithfulness      : {bal_df['bal_faithfulness'].mean():.4f}")
    print(f"  Answer Relevancy  : {bal_df['bal_answer_relevancy'].mean():.4f}")
    print(f"  Context Relevancy : {bal_df['bal_context_relevancy'].mean():.4f}")
except Exception as e:
    print(f"[ERROR] Balanced RAGAS failed: {e}")
    raise


# ── Step 9: Merge all results ─────────────────────────────────
print("\nMerging results ...")

meta_df = pd.DataFrame([{
    "query_id"        : r["query_id"],
    "query"           : r["query"],
    "query_type"      : r["query_type"],
    "privileged_rate" : r["privileged_rate"],
    "srr"             : r["srr"],
    "spd"             : r["spd"],
    "precision_at_10" : r["precision_at_10"],
    "ndcg_at_10"      : r["ndcg_at_10"],
    "mrr"             : r["mrr"],
    "standard_answer" : r["standard_answer"],
    "balanced_answer" : r["balanced_answer"],
} for r in records])

final_df = pd.concat([
    meta_df.reset_index(drop=True),
    std_df[["std_faithfulness","std_answer_relevancy","std_context_relevancy"]].reset_index(drop=True),
    bal_df[["bal_faithfulness","bal_answer_relevancy","bal_context_relevancy"]].reset_index(drop=True),
], axis=1)

# save detailed results
out_detail = os.path.join(OUTPUT_DIR, "ragas_results.csv")
final_df.to_csv(out_detail, index=False)
print(f"✓ Saved: {out_detail}  ({len(final_df)} rows)")


# ── Step 10: Comparison table ─────────────────────────────────
print("\n" + "="*65)
print("  RAGAS: Standard vs Balanced Prompt Comparison")
print("="*65)

comparison_df = pd.DataFrame({
    "Metric"          : ["Faithfulness","Answer Relevancy","Context Relevancy"],
    "Standard Prompt" : [
        round(final_df["std_faithfulness"].mean(), 4),
        round(final_df["std_answer_relevancy"].mean(), 4),
        round(final_df["std_context_relevancy"].mean(), 4),
    ],
    "Balanced Prompt" : [
        round(final_df["bal_faithfulness"].mean(), 4),
        round(final_df["bal_answer_relevancy"].mean(), 4),
        round(final_df["bal_context_relevancy"].mean(), 4),
    ],
})
comparison_df["Difference (Bal-Std)"] = (
    comparison_df["Balanced Prompt"] - comparison_df["Standard Prompt"]
).round(4)

print(comparison_df.to_string(index=False))

out_compare = os.path.join(OUTPUT_DIR, "ragas_comparison_table.csv")
comparison_df.to_csv(out_compare, index=False)
print(f"\n✓ Saved: {out_compare}")


# ── Step 11: Breakdown by privilege group ─────────────────────
print("\nBreakdown by privilege group:")

final_df["priv_group"] = final_df["privileged_rate"].apply(
    lambda x: "High Privileged (>=0.6)"
    if x is not None and float(x) >= 0.6
    else "Low Privileged (<0.6)"
)

priv_agg = (
    final_df.groupby("priv_group")[[
        "std_faithfulness","std_answer_relevancy","std_context_relevancy",
        "bal_faithfulness","bal_answer_relevancy","bal_context_relevancy",
    ]]
    .mean().round(4)
)
print(priv_agg.to_string())

out_priv = os.path.join(OUTPUT_DIR, "ragas_by_privilege.csv")
priv_agg.to_csv(out_priv)
print(f"\n✓ Saved: {out_priv}")


# ── Final summary ─────────────────────────────────────────────
print("\n" + "="*65)
print("  ALL DONE!")
print(f"  Output: {OUTPUT_DIR}")
print("  ├── ragas_results.csv              (per query, all metrics)")
print("  ├── ragas_comparison_table.csv     (std vs balanced)")
print("  ├── ragas_by_privilege.csv         (by privilege group)")
print("  └── ragas_answers_checkpoint.json  (GPT answers backup)")
print("="*65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Google Drive mounted
 All imports successful
 Config ready | Output → /content/drive/MyDrive/arxiv-rag-project/data2/cs_results

Loading baseline_results.json ...
 Loaded 150 records
  Sample keys: ['query_id', 'query', 'query_type', 'method', 'synthesis', 'contexts', 'precision_at_10', 'ndcg_at_10', 'mrr', 'privileged_rate', 'srr', 'spd']
  Sample query: What are the main approaches to transfer learning in deep ne...
  Num contexts: 10

Generating answers ...
  150 queries × 2 prompts = 300 GPT calls



Generating GPT answers: 100%|██████████| 150/150 [21:20<00:00,  8.54s/it]



 Generated 150 answer pairs
✓ Answers saved to checkpoint: /content/drive/MyDrive/arxiv-rag-project/data2/cs_results/ragas_answers_checkpoint.json

Running RAGAS — STANDARD prompt ...


Evaluating:   0%|          | 0/450 [00:00<?, ?it/s]

 Standard RAGAS complete
  Faithfulness      : 0.9329
  Answer Relevancy  : 0.7823
  Context Relevancy : 0.0568

Running RAGAS — BALANCED prompt ...


Evaluating:   0%|          | 0/450 [00:00<?, ?it/s]

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1994' coro=<AsyncClient.aclose() done, defined at /usr/local/lib/python3.12/dist-packages/httpx/_client.py:2011> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/httpx/_client.py", line 2018, in aclose
    await self._transport.aclose()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 385, in aclose
    await self._pool.aclose()
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_async/connection.py", line 173, in aclose
    await self._connection.aclose()
  File "/usr/local/lib/pyt

✓ Balanced RAGAS complete
  Faithfulness      : 0.9617
  Answer Relevancy  : 0.9028
  Context Relevancy : 0.0568

Merging results ...
✓ Saved: /content/drive/MyDrive/arxiv-rag-project/data2/cs_results/ragas_results.csv  (150 rows)

  RAGAS: Standard vs Balanced Prompt Comparison
           Metric  Standard Prompt  Balanced Prompt  Difference (Bal-Std)
     Faithfulness           0.9329           0.9617                0.0288
 Answer Relevancy           0.7823           0.9028                0.1205
Context Relevancy           0.0568           0.0568                0.0000

✓ Saved: /content/drive/MyDrive/arxiv-rag-project/data2/cs_results/ragas_comparison_table.csv

Breakdown by privilege group:
                         std_faithfulness  std_answer_relevancy  std_context_relevancy  bal_faithfulness  bal_answer_relevancy  bal_context_relevancy
priv_group                                                                                                                                          